In [1]:
import pdfplumber
import pandas as pd
import re

pdf_path = "Remittance_Jeronimo_Martins.PDF"

# Documento: números o letras+números
# Importe Pago: último número con separadores de miles
pattern = re.compile(
    r"^([A-Z0-9]+)\s+.*?\s+([\d\.]+)$"
)

rows = []

with pdfplumber.open(pdf_path) as pdf:
    for page in pdf.pages:
        text = page.extract_text()
        if not text:
            continue

        for line in text.split("\n"):
            line = line.strip()

            # Saltar encabezados, totales y líneas basura
            if (
                not line
                or line.startswith("N. Documento")
                or line.startswith("Total")
                or "Página" in line
            ):
                continue

            m = pattern.match(line)
            if m:
                documento = m.group(1)
                importe_pago = m.group(2)

                rows.append({
                    "N. Documento": documento,
                    "Importe Pago": importe_pago
                })

df = pd.DataFrame(rows)

print(df.head())


  N. Documento   Importe Pago
0           AV              7
1          NIT  7704269000018
2   0001944364        165.450
3   0001944365        154.420
4   0001944366        121.330


In [3]:
rutas["remittance"]
import pdfplumber
import pandas as pd
import re

pdf_path = "Remittance_Jeronimo_Martins.PDF"

# Regex:
# 1) Documento (primer token)
# 2) Importe Pago (último número de la línea)
line_pattern = re.compile(
    r"^([A-Z0-9]+)\s+.*?\s+([\d\.]+)$"
)

def es_documento_valido(doc: str) -> bool:
    """
    Documento válido:
    - Solo números (mínimo 6 dígitos)
    - Letras + números (ej: NCMI10344537, PMP1273610)
    """
    return (
        (doc.isdigit() and len(doc) >= 6)
        or bool(re.match(r"^[A-Z]{2,5}\d{5,}$", doc))
    )

rows = []

with pdfplumber.open(rutas["remittance"]) as pdf:
    for page in pdf.pages:
        text = page.extract_text()
        if not text:
            continue

        for line in text.split("\n"):
            line = line.strip()

            # Filtros rápidos de ruido
            if (
                not line
                or line.startswith("N. Documento")
                or line.startswith("Total")
                or "Página" in line
            ):
                continue

            m = line_pattern.match(line)
            if not m:
                continue

            documento = m.group(1)
            importe_pago = m.group(2)

            if not es_documento_valido(documento):
                continue

            rows.append({
                "N. Documento": documento,
                "Importe Pago": importe_pago
            })

df = pd.DataFrame(rows)

print(df.head())
print(f"Registros extraídos: {len(df)}")


  N. Documento Importe Pago
0   0001944364      165.450
1   0001944365      154.420
2   0001944366      121.330
3   0001944367       16.545
4   0001944368      170.903
Registros extraídos: 180


In [4]:
df.head(30)

,N. Documento,Importe Pago
0,0001944364,165.450
1,0001944365,154.420
2,0001944366,121.330
3,0001944367,16.545
4,0001944368,170.903
5,0001944369,110.260
6,0001944370,104.747
7,0001944371,77.182
8,0001944372,16.539
9,0001944373,82.695
